# 📈 Recurrent Neural Networks (RNN) - Introduction & Time Series

**Welcome, class! 👋**

Today, we are moving beyond static images (CNNs) and independent data points. We are entering the world of **Sequences**.

If you think about it, life is a sequence. The sentence you are reading right now only makes sense because you remember the words that came before it. If I just shouted "Network!" at you without the context of "Neural", it would mean something totally different.

Standard Neural Networks (and CNNs) have a major limitation: **They have no memory.** They treat every input as a brand new, isolated event. 

**RNNs (Recurrent Neural Networks)** are different. They have a "loop" that allows information to persist. They are the standard for:
- Time Series Forecasting (Stock prices, Weather)
- Natural Language Processing (Translation, Chatbots)
- Speech Recognition

In this lab, we will build a **Time Series Predictor** from scratch using PyTorch. We will use a synthetic dataset (Simulated Flight Passengers or Sales Data) to ensure everyone has the same clean data to learn from.

### 1. Import Libraries
As always, we need our tools. Notice we are importing `torch.nn`. This contains the `RNN` and `LSTM` layers we need.

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler

# Set random seed for reproducibility (so we all get the same graph!)
torch.manual_seed(42)
np.random.seed(42)

### 2. Creating "Fake" Data 🧪
Instead of downloading a complex CSV file, let's generate a clean **Sine Wave** with some noise. This represents a perfect seasonal pattern (like hourly temperature or yearly sales).

We will generate 1000 data points.

In [ ]:
# Generate 1000 points from 0 to 100
t = np.linspace(0, 100, 1000)

# specific relationship: y = sin(t) + some minor noise
y = np.sin(t) + np.random.normal(0, 0.05, 1000)

plt.figure(figsize=(12, 4))
plt.title("Our Synthetic Time Series Data")
plt.plot(y)
plt.grid(True)
plt.show()

--- 
### 🧠 Thinking Point 1:
Look at the graph above. If I gave you the value at `t=500`, could you predict `t=501`? 
Probably yes, because the pattern is obvious. 

**But what if I only showed you ONE point?** If I just said "The value is 0.5", is the graph going UP or DOWN? You can't know! 

**Conclusion:** You need a *sequence* of past data (Context) to predict the future. We call this the **"Look-back Window"**.

### 3. Data Preprocessing

Neural Networks work best when data is small, typically between -1 and 1 or 0 and 1. We will use `MinMaxScaler`.

In [ ]:
# Normalize data to be between -1 and 1
scaler = MinMaxScaler(feature_range=(-1, 1))
y_normalized = scaler.fit_transform(y.reshape(-1, 1))

# Convert to PyTorch Tensor
y_tensor = torch.FloatTensor(y_normalized).view(-1)

print(f"Original shape: {y.shape}")
print(f"Tensor shape: {y_tensor.shape}")

#### Creating Sequences (The Sliding Window)
We need to organize our data into `(Input, Label)` pairs.

If our window size is 10:
- **Input:** Days 1 to 10
- **Label (Target):** Day 11

Next pair:
- **Input:** Days 2 to 11
- **Label (Target):** Day 12

This creates a dataset where the model learns: "Given these 10 days, what happens next?"

In [ ]:
def create_sequences(input_data, window_size):
    sequences = []
    labels = []
    
    L = len(input_data)
    for i in range(L - window_size):
        # Retrieve the sequence of length 'window_size'
        seq = input_data[i:i+window_size]
        # The target is the very next value after the sequence
        label = input_data[i+window_size]
        
        sequences.append(seq)
        labels.append(label)
        
    return torch.stack(sequences), torch.stack(labels)

# Settings
window_size = 40  # Look back at the last 40 points to predict the next one

X, z = create_sequences(y_tensor, window_size)

print(f"Input shape (X): {X.shape}  <- (Total Sequences, Window Size)")
print(f"Target shape (z): {z.shape}  <- (Total Targets)")

### 4. Train/Test Split
We can't shuffle time series data! (Why? Because the future cannot predict the past in training). We must split sequentially.

In [ ]:
test_size = 100
train_size = len(X) - test_size

train_X = X[:train_size]
train_y = z[:train_size]

test_X = X[train_size:]
test_y = z[train_size:]

print(f"Training sets: {train_X.shape}")
print(f"Testing sets: {test_X.shape}")

### 5. Defining the Model: LSTM (Long Short-Term Memory)

We will use an **LSTM** instead of a basic RNN. 

**Why?** Basic RNNs suffer from "Short-term memory" (Vanishing Gradient problem). LSTMs have internal gates (Forget, Input, Output) that allow them to decide what to remember and what to forget over long sequences.

**Architecture:**
1. **LSTM Layer**: Reads the sequence.
2. **Linear Layer**: Takes the final result of the LSTM and maps it to a single value (our prediction).

In [ ]:
class TimeSeriesLSTM(nn.Module):
    def __init__(self, input_size=1, hidden_size=50, output_size=1):
        super().__init__()
        self.hidden_size = hidden_size
        
        # The LSTM Layer
        # input_size = 1 (we are only passing 1 value per time step: the sine value)
        # hidden_size = 50 (the number of features the LSTM 'learns' to represent the pattern)
        self.lstm = nn.LSTM(input_size, hidden_size, batch_first=True)
        
        # The Linear Layer (Fully Connected)
        # Maps the 50 hidden features to 1 output value
        self.linear = nn.Linear(hidden_size, output_size)
        
    def forward(self, x):
        # x shape: (batch_size, seq_length, input_features)
        # We need to reshape x to have the input_features dimension, usually it is [Batch, Sequence] -> [Batch, Sequence, 1]
        x = x.view(len(x), -1, 1)
        
        # LSTM returns: out, (hidden_state, cell_state)
        # We only care about 'out'
        lstm_out, _ = self.lstm(x)
        
        # We only want the output of the LAST time step in the sequence
        # (Specifically, we want to know what comes AFTER the sequence ends)
        last_time_step = lstm_out[:, -1, :]
        
        # Pass through linear layer to get prediction
        predictions = self.linear(last_time_step)
        
        return predictions

--- 
### 🧠 Thinking Point 2:
In the `forward` function, why did we do `lstm_out[:, -1, :]`?

The LSTM gives us an output for *every* step in the sequence (Day 1, Day 2... Day 40). But we only care about the final accumulated "memory" after seeing all 40 days to predict Day 41. So, we take the last one!

In [ ]:
# Instantiate the model
model = TimeSeriesLSTM()

# Loss Function: MSE (Mean Squared Error) is standard for Regression numbers
criterion = nn.MSELoss()

# Optimizer: Adam is usually the best choice to start with
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

print(model)

### 6. Training Loop
This is standard PyTorch. 100 epochs should be enough for this simple dataset.

In [ ]:
epochs = 60
train_losses = []
test_losses = []

print("Starting Training... 🚀")

for i in range(epochs):
    
    # --- Training Phase ---
    model.train()
    
    optimizer.zero_grad()
    
    y_pred = model(train_X)
    loss = criterion(y_pred.view(-1), train_y) # flatten prediction to match target
    
    loss.backward()
    optimizer.step()
    
    train_losses.append(loss.item())
    
    # --- Evaluation Phase (Optional inside loop, but good for checking overfit) ---
    model.eval()
    with torch.no_grad():
        test_pred = model(test_X)
        test_loss = criterion(test_pred.view(-1), test_y)
        test_losses.append(test_loss.item())

    # Print status every 10 epochs
    if i % 10 == 0:
        print(f'Epoch {i:3} | Train Loss: {loss.item():.5f} | Test Loss: {test_loss.item():.5f}')

print("Training Complete! ✅")

In [ ]:
# Plotting Loss
plt.figure(figsize=(10,5))
plt.plot(train_losses, label='Training Loss')
plt.plot(test_losses, label='Test Loss')
plt.title("Loss over Epochs")
plt.legend()
plt.show()

### 7. Evaluation & Forecasting

Now, let's see how well our model predicts the future. We will visualize the predictions on the Test Set (which the model never saw during training).

In [ ]:
model.eval()
with torch.no_grad():
    # Predict on the test data
    predicted_norm = model(test_X).numpy()

# Inverse transform to get back to original scale (un-normalize)
predicted_real = scaler.inverse_transform(predicted_norm)
actual_real = scaler.inverse_transform(test_y.reshape(-1, 1))

plt.figure(figsize=(12, 5))
plt.title("Prediction vs Reality")
plt.plot(actual_real, label="Actual Data")
plt.plot(predicted_real, label="LSTM Prediction")
plt.legend()
plt.grid(True)
plt.show()

### 🏁 Conclusion

You have just built your first RNN/LSTM!

**Summary of what we did:**
1. Generated sequential data (Sine wave).
2. Created "Windows" of data (Look back 40 days -> Predict Day 41).
3. Fed the sequence into an `nn.LSTM` layer.
4. Used the final hidden state to predict the value.

**Challenge for you:**
Try changing the `window_size` at the top (Seq Length). If you make it very small (e.g., 5), does the model get worse? Why? (Hint: Does 5 days give enough context to know where you are in the sine wave?)